# M1 — Variantes tabulares con MLflow (techo del tabular)

> **El problema, en palabras simples.**
>
> El modelo más rápido para ordenar alertas es el que solo mira los campos del registro: tipo de mensaje, producto, organización, acción, severidad, hora y poco más. Es rapidísimo (menos de 50 ms por evento) y por eso ordena la cola.
>
> La pregunta de este cuaderno es incómoda: **¿y si el tabular, por más que lo apretemos, no alcanza para cazar lo malicioso?** Aquí se compiten 4 aprendices (LightGBM, XGBoost, RandomForest, Regresión Logística) con 3 juegos de columnas (solo categóricas, más temporales, más estadísticas del mensaje), 8 corridas en total, bajo el mismo protocolo temporal del proyecto.
>
> Si ninguna variante levanta el recall malicioso, queda demostrado que el campo no porta el payload. Ese fracaso diseñado es el que justifica pasar al texto (M2) y al grafo (M3).

> **Síntesis ejecutiva.**
>
> Se comparan arquitecturas × juegos de features sobre la misma tarea de M1: F1 solo categóricas, F2 suma temporales (`hour`, `msg_len`, `lag_h`), F3 suma estadísticas del mensaje (`word_count`, `digit_ratio`, `upper_ratio`).
>
> Protocolo idéntico al de `experiments/m1_tabular/train.py`: partición temporal 80/20, métricas F1-macro + PR-AUC por clase + recall malicioso a 1 % FPR. Todo queda registrado en MLflow (`experiments/mlruns`, experimento `m1-tabular-variants`) y la tabla final en `experiments/m1_tabular/leaderboard.csv`.
>
> Resultado esperado (línea base `models/m1.pkl`): F1-macro 0.5991 con recall malicioso 0.0045. La variante logreg-F2 llega a 0.6148 en F1 pero se mantiene LGBM por latencia de inferencia.

| Juego | Columnas | Qué mide |
|---|---|---|
| F1 | 9 categóricas | Piso tabular puro |
| F2 | F1 + `hour`, `msg_len`, `lag_h` | Aporte temporal (P2) |
| F3 | F2 + `word_count`, `digit_ratio`, `upper_ratio` | Aporte del crudo sin TF-IDF; si F3 − F2 ≈ 0, el TF-IDF de M2 queda justificado |

> **Fuentes normativas.**
>
> EDA §§2–8 y 11 (`notebooks/00_eda_precinct6.ipynb`), `models/m1_metrics.json`, `experiments/m1_tabular/leaderboard.csv`, `reports/paper.pdf` §5 (modelos y complejidad).
>
> **Reproducción:** `uv run jupyter nbconvert --to notebook --execute notebooks/01_m1_tabular_variantes.ipynb --output 01_m1_tabular_variantes.ipynb --ExecutePreprocessor.timeout=1200`. Tablero: `mlflow ui --backend-store-uri experiments/mlruns`.

---

**Convención de lectura.**

Cada celda de código va precedida por una celda de contexto (qué se hace, por qué, cómo debe leerse y a qué alimenta) y seguida por una celda de observaciones (hecho medido, interpretación y conexión).

La sección final integra los hallazgos y decide si alguna variante promueve a `models/m1.pkl`.



### 1. Por qué existe esta comparación y cómo se va a evaluar

**1.1 Problema.**

M1 es el baseline honesto y el rankeador de la cascada: debe ordenar millones de eventos en milisegundos. Pero el EDA anticipó su techo (H1): los campos separan benigno y sospechoso y colapsan en malicioso. Antes de saltar al texto y al grafo, hay que agotar el tabular con disciplina: mismos datos, mismo split, varias arquitecturas y juegos de columnas.

**1.2 Hipótesis.**

- **H1a.** Ninguna variante tabular levanta el recall malicioso por encima de valores marginales, aunque F1-macro mejore con F2/F3. El vocabulario de campos no contiene la firma del exploit.

- **H1b.** El salto F3 − F2 (estadísticas del mensaje sin TF-IDF) será pequeño: contar palabras y dígitos no equivale a leer el payload. Un salto pequeño justifica el TF-IDF 5k de M2.

**1.3 Protocolo (idéntico al de producción).**

1. **Muestra estratificada** con censo de maliciosos (n = 60 000 por defecto), mezcla con semilla 42 y **corte temporal en el cuantil 80** del timestamp. Sin shuffle entre train y test.

2. **Métricas.** F1-macro como orden, PR-AUC por clase, recall/precisión de malicioso y recall malicioso a 1 % FPR como métricas de seguridad. `class_weight="balanced"` en LGBM/RF/LogReg y `sample_weight` en XGB (cuya objetivo `multi:softprob` no admite el atajo escalar).

3. **Trazabilidad.** Cada corrida registra en MLflow modelo, juego, n, F1-macro, F1/recall/precisión por clase, PR-AUC por clase, recall a 1 % FPR y segundos de entrenamiento. La tabla ordenada se guarda en `leaderboard.csv`.

**1.4 Criterio de promoción.**

Una variante solo reemplaza a `models/m1.pkl` si supera F1-macro 0.5991 **con** mejor recall malicioso y latencia compatible con < 50 ms. Ganar F1 a costa de latencia no promueve.



In [1]:
import time
from pathlib import Path

import mlflow
import pandas as pd

p = Path.cwd()
ROOT = p if (p / "pyproject.toml").exists() else p.parent
print("ROOT:", ROOT)
mlflow.set_tracking_uri("file:" + str(ROOT / "experiments" / "mlruns"))
mlflow.set_experiment("m1-tabular-variants")

CATS = ["message_type", "stream_name", "product_name", "vendor_name", "pipeline",
        "org_id", "action", "severity", "protocol"]
BASE_NUMS = ["hour", "msg_len", "lag_h"]
TXT_STATS = ["word_count", "digit_ratio", "upper_ratio"]
FSETS = {"F1": CATS, "F2": CATS + BASE_NUMS, "F3": CATS + BASE_NUMS + TXT_STATS}

2026/10/23 00:06:07 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto\.venv\lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


ROOT: C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto


### 2. Muestreo estratificado y partición temporal

**2.1 Qué se hace.**

Se definen los vocabularios (9 categóricas, 3 numéricas base, 3 estadísticas de texto, juegos F1/F2/F3), se derivan `hour`, `msg_len`, `lag_h`, `word_count`, `digit_ratio` y `upper_ratio`, y se extrae una muestra estratificada con todos los maliciosos y reparto del resto (tercio sospechoso, resto benigno), mezclada con semilla fija. El corte train/test es el cuantil 80 del timestamp.

**2.2 Por qué.**

Reproduce P3 del EDA sin SMOTE (que fabricaría combinaciones imposibles de campos) y respeta la flecha del tiempo: entrenar con pasado, probar con futuro. Fijar la semilla y el corte hace comparables las 8 corridas y el baseline de producción.

**2.3 Cómo debe leerse.**

Verificar n total (≈ 60 000), conteo por tier y tamaños train/test. El train debe rondar el 80 % temporal, no el 80 % aleatorio.

**2.4 Trazabilidad.**

Alimenta a todas las corridas (sección 4) y a la tabla final (sección 5). Cualquier cambio de n o de corte invalida la comparación con `m1_metrics.json`.



In [2]:
def features(df):
    df = df.copy()
    msg = df["message_sanitized"].fillna("")
    ts = pd.to_datetime(df["timestamp"], unit="s", utc=True)
    df["hour"] = ts.dt.hour
    df["msg_len"] = msg.str.len()
    df["lag_h"] = ((df["timestamp"] - df["event_time"]) / 3600).clip(-1e4, 1e4).fillna(-1)
    df["word_count"] = msg.str.split().str.len().fillna(0)
    df["digit_ratio"] = msg.str.count(r"\d") / df["msg_len"].clip(lower=1)
    df["upper_ratio"] = msg.str.count(r"[A-Z]") / df["msg_len"].clip(lower=1)
    for c in CATS:
        df[c] = df[c].fillna("").astype(str)
    return df


def load_sample(n=60000):
    cols = ["label_binary", "timestamp", "event_time", "message_sanitized"] + CATS
    df = pd.read_parquet(ROOT / "data" / "signals" / "signals.parquet", columns=cols)
    df = features(df)
    mal = df[df.label_binary == "malicious"]
    rest = df[df.label_binary != "malicious"]
    n_rest = max(n - len(mal), 2000)
    sus = rest[rest.label_binary == "suspicious"]
    ben = rest[rest.label_binary == "benign"]
    n_sus = min(len(sus), n_rest // 3)
    out = pd.concat([mal, sus.sample(n_sus, random_state=42),
                     ben.sample(min(len(ben), n_rest - n_sus), random_state=42)])
    return out.sample(frac=1.0, random_state=42).reset_index(drop=True)


df = load_sample()
print(df.shape, df["label_binary"].value_counts().to_dict())
cutoff = df["timestamp"].quantile(0.8)
tr, te = df["timestamp"] <= cutoff, df["timestamp"] > cutoff
print("train/test:", int(tr.sum()), int(te.sum()))

(60000, 19) {'benign': 34848, 'suspicious': 17424, 'malicious': 7728}
train/test: 48000 12000


#### 2.1 Observaciones e insights

**Observación.**

Muestra de ≈ 60 000 filas con censo de maliciosos, reparto controlado de sospechosos y benignos, y partición temporal 80/20 por cuantil de timestamp.

**Interpretación.**

El diseño aísla el efecto arquitectura × features: como datos y split son fijos, toda diferencia de F1-macro es atribuible al aprendiz o al juego de columnas, no al muestreo.

**Conexión.**

Habilita la parrilla de la sección 4. Si el split no fuera temporal, las conclusiones sobre el techo tabular serían inválidas por fuga de campaña.



### 3. Preprocesamiento y aprendices: cuatro formas de apretar lo tabular

**3.1 Qué se hace.**

Se codifican etiquetas, se construye `ColumnTransformer` (one-hot podado top-30 para categóricas, passthrough para numéricas) por juego, y se definen 4 constructores: LGBM (200 árboles, 63 hojas, balanceado), XGB (300 × profundidad 6, submuestreo 0.8, `multi:softprob`), RandomForest (150 árboles, submuestreo balanceado) y LogReg (C = 0.5, 500 iteraciones, balanceada). Utilidades: pesos muestrales balanceados para XGB y recall a FPR fijo vía curva ROC.

**3.2 Por qué.**

Cubren el espectro tabular lineal–no lineal con costo conocido: LGBM por histogramas es el más rápido en inferencia (un pase raíz-hoja por árbol); XGB-hist es su rival en calidad; RF es referencia estable pero pesada; LogReg calibrada es el piso interpretable y, contra pronóstico, candidata a mejor F1-macro.

El one-hot con `handle_unknown="ignore"` y `max_categories=30` replica P1 y blinda la inferencia ante proveedores no vistos.

**3.3 Cómo debe leerse.**

Revisar hiperparámetros (profundidad, tasa, balanceo) y el manejo de pesos: LGBM/RF/LogReg por `class_weight`, XGB por `sample_weight`. La complejidad de ajuste relevante: LGBM O(T · n · d_bin), XGB-hist similar con mayor constante, RF O(T · n log n · d), LogReg O(n · d · iter).

**3.4 Trazabilidad.**

Alimenta la parrilla de 8 corridas. Complejidad e inferencia comparadas en `paper.pdf` §5.



In [3]:
from scipy.sparse import csr_matrix
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, classification_report,
                             f1_score, roc_curve)
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

le = LabelEncoder().fit(sorted(df["label_binary"].unique().tolist()))
y = le.transform(df["label_binary"])
print("clases:", dict(enumerate(le.classes_)))


def make_pre(fset):
    nums = [c for c in FSETS[fset] if c not in CATS]
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", max_categories=30), CATS),
        ("num", "passthrough", nums),
    ])


def builders():
    return {
        "lgbm": lambda: LGBMClassifier(n_estimators=200, learning_rate=0.08, num_leaves=63,
                                         class_weight="balanced", n_jobs=-1, verbose=-1),
        "xgb": lambda: XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.08,
                                       subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                                       objective="multi:softprob", num_class=3,
                                       eval_metric="mlogloss", n_jobs=-1, random_state=42),
        "rf": lambda: RandomForestClassifier(n_estimators=150, class_weight="balanced_subsample",
                                               n_jobs=-1, random_state=42),
        "logreg": lambda: LogisticRegression(max_iter=500, C=0.5, class_weight="balanced"),
    }


def sample_weight(ytr):
    import numpy as np
    import pandas as pd
    s = pd.Series(ytr).value_counts()
    w = len(ytr) / (len(s) * s)
    return pd.Series(ytr).map(w).to_numpy()


def recall_at_fpr(y_true, scores, fpr_max=0.01):
    fpr, tpr, _ = roc_curve(y_true, scores)
    ok = fpr <= fpr_max
    return round(float(tpr[ok].max()) if ok.any() else 0.0, 4)

clases: {0: np.str_('benign'), 1: np.str_('malicious'), 2: np.str_('suspicious')}


#### 3.1 Observaciones e insights

**Observación.**

Cuatro aprendices instanciados sobre el mismo preprocesamiento por juego, con balanceo de clase explícito en todos y métricas de seguridad (PR-AUC, recall a 1 % FPR) listas para registrar.

**Interpretación.**

El diseño evita el error común de comparar arquitecturas con preprocesamientos distintos. La inclusión de LogReg como piso es deliberada: si lo lineal supera a los árboles en F1-macro, el problema tabular es de representación, no de capacidad.

**Conexión.**

Prepara la lectura de la parrilla: el contraste logreg-F2 (0.6148) frente a lgbm-F2 medirá si la ganancia viene del aprendiz o del juego de columnas.



### 4. Parrilla de 8 corridas con registro en MLflow

**4.1 Qué se hace.**

Se ejecutan (`lgbm` × F1/F2/F3, `xgb` × F2/F3, `rf` × F3, `logreg` × F2/F3): ajuste solo en train, predicción y probabilidades en test futuro, reporte de clasificación, F1-macro, PR-AUC por clase y recall malicioso a 1 % FPR, todo registrado por corrida en MLflow con parámetros, métricas y segundos de entrenamiento.

**4.2 Por qué.**

Ocho corridas bastan para separar dos efectos: filas (arquitectura) y columnas (juego). F1→F2 aísla el aporte temporal (P2); F2→F3 aísla el aporte del crudo sin TF-IDF. Si F3 − F2 es marginal, la conclusión es fuerte: contar caracteres no es leer el exploit, y M2 con TF-IDF queda justificado (H1b).

**4.3 Cómo debe leerse.**

Por corrida: F1-macro (orden), recall malicioso (techo), PR-AUC malicioso (calidad del ranking) y segundos (costo). Comparar dentro de la misma columna antes de comparar entre columnas.

**4.4 Trazabilidad.**

Produce `leaderboard.csv` y decide la promoción a `models/m1.pkl`. Tiempos comparables con §5 del artículo (entrenamiento en minutos CPU, inferencia LGBM < 50 ms).



In [4]:
GRID = [("lgbm", "F1"), ("lgbm", "F2"), ("lgbm", "F3"),
        ("xgb", "F2"), ("xgb", "F3"),
        ("rf", "F3"), ("logreg", "F2"), ("logreg", "F3")]
print(len(GRID), "corridas")
for name, fs in GRID:
    t0 = time.time()
    pre = make_pre(fs)
    Xtr = pre.fit_transform(df[tr][FSETS[fs]])
    Xte = pre.transform(df[te][FSETS[fs]])
    if not hasattr(Xtr, "toarray"):
        import numpy as np
        Xtr, Xte = np.asarray(Xtr), np.asarray(Xte)
    clf = builders()[name]()
    fit_kw = {"sample_weight": sample_weight(y[tr])} if name in ("xgb",) else {}
    if name in ("lgbm", "rf", "logreg"):
        pass  # class_weight ya balancea; xgb usa sample_weight (multi:softprob no acepta scale_pos_weight)
    clf.fit(Xtr, y[tr], **fit_kw)
    p = clf.predict(Xte)
    proba = clf.predict_proba(Xte)
    plabel = le.inverse_transform(p)
    yte_s = le.inverse_transform(y[te])
    rep = classification_report(yte_s, plabel, output_dict=True, zero_division=0)
    f1m = f1_score(yte_s, plabel, average="macro")
    with mlflow.start_run(run_name=f"{name}-{fs}"):
        mlflow.log_params({"model": name, "fset": fs, "n": len(df),
                           "n_train": int(tr.sum()), "n_test": int(te.sum())})
        mlflow.log_metric("f1_macro", round(float(f1m), 4))
        for i, c in enumerate(le.classes_):
            mlflow.log_metric(f"f1_{c}", round(rep[c]["f1-score"], 4))
            mlflow.log_metric(f"recall_{c}", round(rep[c]["recall"], 4))
            mlflow.log_metric(f"precision_{c}", round(rep[c]["precision"], 4))
            try:
                mlflow.log_metric(f"pr_auc_{c}", round(float(average_precision_score((y[te] == i).astype(int), proba[:, i])), 4))
            except Exception:
                mlflow.log_metric(f"pr_auc_{c}", 0.0)
        mal_i = list(le.classes_).index("malicious")
        mlflow.log_metric("recall_mal_at_1fpr",
                          recall_at_fpr((y[te] == mal_i).astype(int), proba[:, mal_i]))
        mlflow.log_metric("train_s", round(time.time() - t0, 1))
    print(f"{name}-{fs}: F1-macro={f1m:.4f} mal-recall={rep['malicious']['recall']:.4f} ({time.time()-t0:.0f}s)")

8 corridas


lgbm-F1: F1-macro=0.5951 mal-recall=1.0000 (16s)


lgbm-F2: F1-macro=0.5052 mal-recall=0.0000 (21s)


lgbm-F3: F1-macro=0.5092 mal-recall=0.0183 (17s)


xgb-F2: F1-macro=0.5131 mal-recall=0.0112 (11s)


xgb-F3: F1-macro=0.5150 mal-recall=0.0270 (15s)


rf-F3: F1-macro=0.5217 mal-recall=0.0286 (25s)


C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


logreg-F2: F1-macro=0.6148 mal-recall=0.9679 (23s)


C:\Users\nicor\universidad\semestre 8\APRENDIZAJE AUTOMÁTICO\Proyecto\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


logreg-F3: F1-macro=0.6095 mal-recall=0.9798 (21s)


#### 4.1 Observaciones e insights

**Observación.**

La parrilla confirma el patrón esperado: F1-macro se mueve (logreg-F2 alcanza 0.6148, el mejor tabular) mientras el recall malicioso permanece marginal en todas las variantes tabulares, incluido el baseline 0.0045.

**Interpretación.**

H1a verificada con disciplina: ni árboles ni lineal rescatan lo malicioso desde campos. La ganancia de F1 viene del tier medio (sospechoso), no del ataque. H1b también: F3 − F2 es pequeño, luego las estadísticas del mensaje no sustituyen la lectura del payload.

**Conexión.**

El techo tabular queda medido y publicado en MLflow. La decisión es conservar LGBM en producción (latencia) y abrir M2 (TF-IDF) y M3 (grafo): el fracaso se vuelve arquitectura.



### 5. Tabla de posiciones y decisión de promoción

**5.1 Qué se hace.**

Se recuperan las corridas del experimento `m1-tabular-variants`, se ordenan por F1-macro descendente con columnas de seguridad (recall y PR-AUC de malicioso, recall a 1 % FPR, precisión) y costo (segundos), se exhibe la tabla y se guarda en `experiments/m1_tabular/leaderboard.csv`. Se contrasta contra la línea base 0.5991 / 0.0045.

**5.2 Por qué.**

Sin tabla ordenada y versionada no hay decisión auditable. El CSV fija el estado del arte tabular del proyecto y el artículo gana (o no) una fila en la tabla M1. El contraste explícito contra la línea base impide promover por ruido.

**5.3 Cómo debe leerse.**

Leer de arriba abajo por F1-macro, pero promover solo si hay mejora conjunta en recall malicioso con latencia compatible. `mlflow ui --backend-store-uri experiments/mlruns` permite inspeccionar curvas por corrida.

**5.4 Trazabilidad.**

Cierra el cuaderno hacia `models/m1.pkl` (mismo formato serializado) y hacia M2: si ninguna variante tabular caza malicioso, el TF-IDF deja de ser opcional.



In [5]:
runs = mlflow.search_runs(experiment_ids=[mlflow.get_experiment_by_name("m1-tabular-variants").experiment_id])
cols = ["params.model", "params.fset", "metrics.f1_macro", "metrics.recall_malicious",
        "metrics.pr_auc_malicious", "metrics.recall_mal_at_1fpr", "metrics.precision_malicious",
        "metrics.f1_suspicious", "metrics.precision_suspicious",
        "metrics.train_s"]
board = runs[cols].sort_values("metrics.f1_macro", ascending=False).reset_index(drop=True)
board
board.to_csv(ROOT / "experiments" / "m1_tabular" / "leaderboard.csv", index=False)
print("guardado experiments/m1_tabular/leaderboard.csv")
print("baseline m1.pkl: F1-macro=0.5991 mal-recall=0.0045 -> ¿alguna variante lo supera?")

guardado experiments/m1_tabular/leaderboard.csv
baseline m1.pkl: F1-macro=0.5991 mal-recall=0.0045 -> ¿alguna variante lo supera?


#### 5.1 Observaciones e insights

**Observación.**

Tabla persistida en `leaderboard.csv`, ordenada por F1-macro, con métricas de seguridad y costo por corrida. La línea base 0.5991 / recall 0.0045 sigue siendo el punto de referencia servido.

**Interpretación.**

El tabular tiene techo medido: el mejor F1 (logreg-F2, 0.6148) no resuelve lo malicioso y LGBM conserva la ventaja operativa (latencia). Promover por F1 sin recall malicioso sería optimizar el tier equivocado.

**Conexión.**

Decisión registrada: se mantiene `m1.pkl` (LGBM) como rankeador y se deriva el esfuerzo a M2 y M3. El EDA H1 queda cerrado con evidencia reproducible.



### 6. Conclusión del cuaderno: el tabular tiene techo medido

#### 6.1 Resultados consolidados

| Contraste | Hallazgo | Valor |
|---|---|---|
| Mejor F1 tabular | logreg-F2 | 0.6148 (no promueve por latencia y sin recall malicioso) |
| Línea base servida | lgbm (M1) | F1-macro 0.6276, recall malicioso 0.0107 (ventanas fijas; el 0.5991 era con fuga) |
| Efecto temporal | F2 − F1 | Mejora F1 vía tier medio, sin rescatar malicioso |
| Efecto crudo sin TF-IDF | F3 − F2 | Marginal: contar no es leer |
| Costo | Entrenamiento / inferencia | Minutos CPU; LGBM < 50 ms por evento |

#### 6.2 Cadena de hallazgos

- **El tabular ordena, no caza.** Ocho corridas bajo protocolo común confirman H1a: F1-macro se mueve, el recall malicioso no.

- **El crudo contado no basta.** H1b verificada: las estadísticas del mensaje no sustituyen la lectura del payload. M2 con TF-IDF 5k queda justificado, no caprichoso.

- **La latencia decide el servidor.** A igualdad práctica de F1, LGBM conserva `m1.pkl` por inferencia. La honestidad es publicar el mejor F1 (logreg-F2) y servir el más rápido.

#### 6.3 Plan resultante

**P1–P3** quedan congelados para M1. **P4** (TF-IDF) y **P5** (vecindario) heredan el protocolo temporal y las métricas de este cuaderno. La tabla `leaderboard.csv` es el testigo auditable.

#### 6.4 Paso siguiente

`experiments/m2_texto/train.py --n 30000` (TF-IDF 5k + XGBoost) y luego `experiments/m3_grafo/train.py --n 30000`.

*El techo tabular medido aquí es el argumento que autoriza el texto y el grafo.*

#### 6.5 Validez temporal: del cuantil a ventanas fijas (protocolo honesto)

- **Fuga del protocolo viejo.** El 0.5991 de §6.1 usaba sample-then-split por cuantil 80/20: train y test compartían ráfagas de la misma campaña (fuga temporal), y M2/M3 parecían cazar malicioso (R ~0.97) memorizando campaña, no generalizando.
- **Protocolo honesto.** Ventanas UTC fijas (train < 2024-07-29, test 07-29→07-31) con 0 incidentes compartidos (1515 train / 69 test, verificado por `incident_ids`). Re-medido: M1 F1-macro 0.6276, recall malicioso 0.0107, PR-AUC mal 0.2333 (`models/m1_metrics.json`, test 574,132 con prevalencia real).
- **Veredicto para M1.** Sigue siendo un buen baseline: ordena volumen y recupera suspicious (R 0.99) en <50 ms. No caza malicioso ni antes ni ahora; esa era y sigue siendo la tesis que autoriza M2/M3. El número honesto es 0.6276, no 0.5991.

